# Lint your Langfuse agent traces with tracelint

You already collect your agent's traces in **Langfuse**. [**tracelint**](https://github.com/AshwinUgale/tracelint) runs *on top of* them: it reads the observations you have and reports **structural** defects — ignored tool errors, schema-violating calls, hallucinated arguments, loops, **duplicate side effects** — each with the exact observation as evidence and a CI exit code.

**No second model judges the trace.** For this class of bug a judge is the wrong tool, because the defect is *decidable by looking at the trace*. Deterministic, reproducible, and ~free.

## Install

```bash
pip install tracelint langfuse
```

## On your live Langfuse traces

Fetch a trace with the Langfuse SDK and hand it straight to tracelint — the adapter accepts the SDK object (or the raw API JSON):

In [ ]:
from langfuse import Langfuse
from tracelint import ToolRegistry, lint_langfuse_trace, render_report

langfuse = Langfuse()                            # reads your LANGFUSE_* env vars
trace = langfuse.api.trace.get("your-trace-id")  # method varies by SDK version

registry = ToolRegistry.load("tools.json")       # optional: side_effecting / failure_when
print(render_report(lint_langfuse_trace(trace, registry=registry), include_candidates=True))

Tool observations that don't use the v4 `tool` type are still recognized if you pass `tool_names={...}` (usually the same names as your `tools.json`).

## Run it now — offline, no Langfuse project or API key

This is an **illustrative** Langfuse trace (constructed, not a captured run) — swap in your own fetched trace from the cell above to lint real runs. Here the support agent is asked to refund order `A100`, the `get_order` lookup **errors**, and the agent refunds the card anyway — using the errored order id — and refunds it **twice**.

In [ ]:
from tracelint import ToolRegistry, lint_langfuse_trace, render_report

# The trace-level `input` (the user's request) seeds provenance, so R3 sees the order id came
# from the user and does not flag it.
trace = {
    "id": "support-run",
    "input": "Refund order A100 to the card on file.",
    "output": "Your refund is processed.",
    "observations": [
        {"id": "o1", "type": "tool", "name": "get_order", "input": {"order_id": "A100"},
         "output": {"order_id": "A100", "status": "error"}, "level": "ERROR",
         "statusMessage": "500 internal error", "startTime": "2024-06-01T10:00:01Z"},
        {"id": "o2", "type": "tool", "name": "refund_order", "input": {"order_id": "A100"},
         "output": {"refunded": True}, "startTime": "2024-06-01T10:00:02Z"},
        {"id": "o3", "type": "tool", "name": "refund_order", "input": {"order_id": "A100"},
         "output": {"refunded": True}, "startTime": "2024-06-01T10:00:03Z"},
    ],
}

# The operator's tools.json: refund_order mutates the world, and {"refunded": false} is its
# declared failure. Declared once, never guessed from the tool name.
registry = ToolRegistry.from_dict({"tools": {
    "get_order": {},
    "refund_order": {"metadata": {"side_effecting": True,
                                  "failure_when": {"pointer": "/refunded", "equals": False}}}}})

report = lint_langfuse_trace(trace, registry=registry)
print(render_report(report, include_candidates=True))
print("\nexit code:", report.exit_code, " (2 = a hard defect; fails CI)")

You'll see three findings, each pointing at the exact observations:

- **R2a `hard_event`** — `get_order` returned an error (read from Langfuse's `level == "ERROR"`).
- **R2b `hard_defect`** — the errored order id was **reused** as the argument to a side-effecting `refund_order`. Data from a failed call fed into a real-world action, no fallback. This is the tier that **fails CI** (exit `2`).
- **R8 `hard_event`** — `refund_order` was called **twice** with the same arguments after the first succeeded: a **double refund**.

It also discloses what it *couldn't* check (no schema for these tools → R1 suppressed) plus per-rule **verification coverage**, so a clean report is honestly clean — not just empty.

## In CI

```bash
tracelint check trace.json --format langfuse --tools tools.json
```

Exit `2` on a hard defect fails the build. Or drop in the GitHub Action:

```yaml
- uses: AshwinUgale/tracelint@v0.5.0
  with:
    path: trace.json
    format: langfuse
    tools: tools.json
```

Repo & docs: **https://github.com/AshwinUgale/tracelint**